In [1]:
from diffrax import (
    AbstractSolver,
    AbstractStepSizeController,
    Heun,
    Tsit5,
    PIDController,
    diffeqsolve,
    ODETerm,
    Event,
    SaveAt
)
from jax import numpy as jnp

In [2]:
# pseudo-code (JAX/Diffrax style), matching your signature
def loss_y(y, target, lam): 
    return 0.5*(y - target)**2 + 0.5*lam*y**2

def neg_activity_grad(t, y, args):
    (target, lam, *_rest) = args
    g = (1 + lam) * y - target      # here 1+lam = 1.1, target = 2.0
    return -g


def steady_state_event_with_timeout(t, y, args, **kwargs):
    target, lam, tol, t_timeout, *_ = args
    grad = (1.0 + lam) * y - target
    grad_resid = jnp.abs(grad) - tol     # triggers when <= 0
    time_resid = (t_timeout - t)         # triggers when <= 0
    return jnp.logical_or(grad_resid <= 0, time_resid <= 0)


# pack args to mirror your call
args = (2.0, 0.1, 1e-3, 10.0, None, None)   # target, lam, tol, timeout, ...
controller = PIDController(
    rtol=1e-4, atol=1e-6,
    dtmin=1e-6,           # avoid vanishing dt
    dtmax=0.25,           # <-- cap step growth to prevent 'inf' near steady state
    safety=0.9, factormin=0.2, factormax=2.0
)

solution = diffeqsolve(
    terms=ODETerm(neg_activity_grad),
    solver=Heun(),
    t0=0.0,
    t1=20.0,
    dt0=1e-2,
    y0=5.0,
    args=args,
    stepsize_controller=controller,
    event=Event(steady_state_event_with_timeout),
    saveat=SaveAt(t0=True, t1=True, steps=True),
)
y_star = solution.ys[-1]   # ≈ 1.8182


In [3]:
solution.ys[:200]

Array([5.       , 4.965193 , 4.9153323, 4.8661637, 4.817634 , 4.7697353,
       4.7224603, 4.6758013, 4.62975  , 4.584299 , 4.5394416, 4.49517  ,
       4.4514775, 4.4083567, 4.3658004, 4.3238015, 4.2823544, 4.2414513,
       4.2010856, 4.1612515, 4.121942 , 4.083151 , 4.0448713, 4.007098 ,
       3.9698248, 3.9330451, 3.8967533, 3.860943 , 3.825609 , 3.790745 ,
       3.7563457, 3.7224057, 3.688919 , 3.6558805, 3.6232848, 3.5911267,
       3.5594008, 3.5281022, 3.4972258, 3.4667664, 3.436719 , 3.4070787,
       3.377841 , 3.349001 , 3.320554 , 3.2924955, 3.2648208, 3.2375255,
       3.2106051, 3.1840553, 3.157872 , 3.1320505, 3.106587 , 3.0814772,
       3.056717 , 3.0323024, 3.0082295, 2.9844942, 2.9610927, 2.9380217,
       2.9152768, 2.8928545, 2.870751 , 2.8489628, 2.8274868, 2.8063192,
       2.7854567, 2.7648954, 2.7446325, 2.7246644, 2.704988 , 2.6856   ,
       2.6664975, 2.647677 , 2.6291354, 2.6108704, 2.5928783, 2.5751567,
       2.5577025, 2.5405126, 2.5235844, 2.506915 , 